In [1]:
import pandas as pd

In [18]:
# 5_1_acoustic_descriptive
df = pd.read_csv("../outputs/analysis/tables/5_1_acoustic_descriptive.csv")
print("5_1_acoustic_descriptive")
print(f"shape: {df.shape}")
print(df.head(10))
print("\naggregated by phoneme (mean of CV)")
print(df.groupby("phoneme")["cv_hz"].mean().sort_values(ascending=False))

# Q2: 
# 5_1_variance_decomposition
vd = pd.read_csv("../outputs/analysis/tables/5_1_variance_decomposition.csv")
print("\n5_1_variance_decomposition")
vd = (vd[["phoneme", "prop_inter", "prop_intra", "prop_resid", "n_tokens"]]
         .sort_values("prop_inter", ascending=False))
print("=== Q2: Variance decomposition of F1 (Lobanov), per phoneme ===")
print(vd.to_string(index=False))

# comparing with neural
lme = pd.read_csv("../outputs/analysis/tables/7_lme_results.csv")

icc_null = lme[(lme["model"] == "null_model") & lme["icc"].notna()].copy()

def parse_tag(tag):
    parts = str(tag).split("_")
    if len(parts) < 2:
        return None, None, None
    phoneme = parts[-1]
    if "acoustic" in tag:
        if phoneme == "all":
            return "acoustic", parts[1], "all"
        return "acoustic", parts[1], phoneme
    if "whisper_l28" in tag:
        return "whisper_l28", "_".join(parts[2:-1]), phoneme
    if "whisper_l8" in tag:
        return "whisper_l8", "_".join(parts[2:-1]), phoneme
    if "xlsr" in tag:
        return f"xlsr_l{parts[1][1:]}", "_".join(parts[2:-1]), phoneme
    return None, None, None

icc_null[["representation", "feature", "phoneme"]] = icc_null["tag"].apply(
    lambda t: pd.Series(parse_tag(t))
)

ac_icc = (icc_null[(icc_null["representation"] == "acoustic")
                   & (icc_null["feature"] == "F1")
                   & (icc_null["phoneme"] != "all")]
          [["phoneme", "icc"]]
          .rename(columns={"icc": "icc_acoustic_F1"}))

neural_icc = icc_null[
    (icc_null["feature"] == "pc_0") & (icc_null["phoneme"] != "all")
].copy()

neural_pivot = (neural_icc.pivot_table(index="phoneme",
                                       columns="representation",
                                       values="icc")
                .add_prefix("icc_"))


combined = ac_icc.merge(neural_pivot, on="phoneme", how="outer")
combined = combined.sort_values("icc_acoustic_F1", ascending=False)
neural_cols = ["icc_whisper_l28", "icc_whisper_l8",
               "icc_xlsr_l4", "icc_xlsr_l12", "icc_xlsr_l20"]

ranked = combined.sort_values("icc_whisper_l28", ascending=False)
print(ranked[["phoneme"] + neural_cols].to_string(index=False))

5_1_acoustic_descriptive
shape: (80, 11)
   Unnamed: 0 phoneme l1_status gender     feature     n      mean    median  \
0           0       a        L1      f  F1_mid_lob   255  0.950496  0.997795   
1           1       a        L1      f  F2_mid_lob   255 -0.437999 -0.832521   
2           2       a        L1      m  F1_mid_lob   865  0.802874  0.726869   
3           3       a        L1      m  F2_mid_lob   865 -0.384206 -0.599827   
4           4       a        L2      f  F1_mid_lob  1272  0.775751  0.852812   
5           5       a        L2      f  F2_mid_lob  1272 -0.510027 -0.615893   
6           6       a        L2      m  F1_mid_lob   253  0.942720  0.982379   
7           7       a        L2      m  F2_mid_lob   253 -0.453767 -0.539811   
8           8       e        L1      f  F1_mid_lob    24 -0.384350 -0.631612   
9           9       e        L1      f  F2_mid_lob    24  0.813287  0.798400   

        std       iqr     cv_hz  
0  0.720002  0.796600  0.141723  
1  0.70786